In [19]:
import numpy as np
import pandas as pd

# 1. Load the dataset
# Adjust the path to your actual file location
df = pd.read_csv('green cleaned all firms.csv', dtype={'ncusip': 'string'})

In [20]:
# 2. Pre-processing
# Ensure date is datetime objects
df['datadate'] = pd.to_datetime(df['datadate'])

# CRITICAL: Verify your Market Cap column name
# Common names in Green datasets: 'mve', 'mktcap', 'me', 'market_equity'
# Replace 'mve' below with the actual column name from your CSV
mkt_col = 'mktcap' 

# 3. Filter for Common Shares only (Standard Academic Practice)
# Keep only shrcd 10 and 11 (Common stocks)
# Use .copy() to avoid SettingWithCopy warnings later
universe = df[df['shrcd'].isin([10, 11])].copy()

# 4. Define the Function to Calculate Breakpoints
def get_nyse_breakpoints(group):
    # Select only NYSE stocks (Exchange Code 1) for the breakpoints
    nyse_stocks = group[group['exchcd'] == 1]
    
    # Check if we have enough NYSE stocks to calculate stats (avoid errors in early years)
    if len(nyse_stocks) < 10:
        return pd.Series({'p20': float('nan'), 'p50': float('nan')})
    
    # Calculate 20th and 50th percentiles
    return pd.Series({
        'p20': nyse_stocks[mkt_col].quantile(0.20),
        'p50': nyse_stocks[mkt_col].quantile(0.50)
    })

# 5. Calculate Breakpoints for Each Month
breakpoints = universe.groupby('datadate').apply(get_nyse_breakpoints).reset_index()

# 6. Merge Breakpoints back to the main data
universe = pd.merge(universe, breakpoints, on='datadate', how='left')

# 7. Apply the Small Cap Filter
# Keep stocks where Market Cap is between the NYSE 20th and 50th percentiles
small_caps = universe[
    (universe[mkt_col] >= universe['p20']) & 
    (universe[mkt_col] < universe['p50'])
].copy()

# Clean up helper columns
small_caps = small_caps.drop(columns=['p20', 'p50'])

print(f"Small Cap Universe filtered. New shape: {small_caps.shape}")
print(small_caps.head())

/var/folders/w3/6myr5tjs6ls3zn1g3ywbc29r0000gn/T/ipykernel_1117/2071772352.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  breakpoints = universe.groupby('datadate').apply(get_nyse_breakpoints).reset_index()


Small Cap Universe filtered. New shape: (552016, 165)
      datadate  permno                comnam    ncusip  shrcd  exchcd  siccd  \
934 1982-10-31   10006  A C F INDUSTRIES INC  00080010     10       1   3743   
935 1982-11-30   10006  A C F INDUSTRIES INC  00080010     10       1   3743   
936 1982-12-31   10006  A C F INDUSTRIES INC  00080010     10       1   3743   
937 1983-01-31   10006  A C F INDUSTRIES INC  00080010     10       1   3743   
938 1983-02-28   10006  A C F INDUSTRIES INC  00080010     10       1   3743   

    industry ticker  gvkey_x  ...  zerotrade      BETA    betasq      rsq1  \
934    Manuf    ACF   1010.0  ...  -0.390607  0.053903  1.244529  0.429830   
935    Manuf    ACF   1010.0  ...  -0.341552 -0.004561  1.161431  0.392651   
936    Manuf    ACF   1010.0  ...  -0.438696 -0.023103  1.135611  0.385028   
937    Manuf    ACF   1010.0  ...  -0.411918  0.010242  1.185947  0.393112   
938    Manuf    ACF   1010.0  ...  -0.437918 -0.034875  1.130314  0.381271 

In [21]:
variables = ['agr', 'bm', 'mom12m', 'mve', 'operprof', 'roeq', 'absacc', 'acc', 'aeavol', 'age', 'baspread', 'BETA', 'bm_ia', 
             'cash', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chfeps', 'chinv', 'chmom', 'chnanalyst', 
             'chpmia', 'chtx', 'cinvest', 'convind', 'currat', 'depr', 'disp', 'divi', 'divo', 'dy', 'ear', 'egr', 'ep', 'fgr5yr', 
             'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'idiovol', 'ill', 'indmom', 
             'invest', 'IPO', 'lev', 'mom1m', 'mom36m', 'ms', 
             'mve_ia', 'nanalyst', 'nincr', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchsale_pchinvt', 
             'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'pricedelay', 'ps', 'rd', 'rd_mve', 'rd_sale', 'realestate', 
             'retvol', 'roaq', 'roavol', 'roic', 'rsup', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sfe', 'sgr', 'sin', 
             'sp', 'std_dolvol', 'std_turn', 'stdcf', 'sue', 'tang', 'tb', 'turn', 'zerotrade']

In [22]:
def preprocess_multiple_characteristics(df, char_cols, time_col='datadate'):
    def process_group(group):
        processed = {}
        for col in char_cols:
            x = group[col].copy()

            if x.dropna().empty:
                # All values missing: impute with 0
                processed[col] = pd.Series(0, index=group.index)
                #print(col)
                continue

            # Winsorize
            p1, p99 = np.percentile(x.dropna(), [1, 99])
            x = x.clip(lower=p1, upper=p99)

            # Standardize
            mean = x.mean()
            std = x.std(ddof=0)
            if std == 0 or np.isnan(std):
                x = x - mean  # Avoid division by zero
            else:
                x = (x - mean) / std

            # Impute missing with 0
            x = x.fillna(0)

            processed[col] = x
        return pd.DataFrame(processed, index=group.index)

    return df.groupby(time_col, group_keys=False).apply(process_group)

df_processed = preprocess_multiple_characteristics(small_caps, variables, time_col='datadate')

/var/folders/w3/6myr5tjs6ls3zn1g3ywbc29r0000gn/T/ipykernel_1117/3356932503.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(time_col, group_keys=False).apply(process_group)


In [23]:
small_caps[variables] = df_processed

In [29]:
small_caps

,datadate,permno,comnam,ncusip,shrcd,exchcd,siccd,industry,ticker,gvkey_x,...,zerotrade,BETA,betasq,rsq1,pricedelay,idiovol,year,mom6,indmom,industry_return
934,1982-10-31,10006,A C F INDUSTRIES INC,00080010,10,1,3743,Manuf,ACF,1010.0,...,-0.958207,-0.008876,1.244529,0.429830,0.124538,-1.397551,1982,-0.067302,-0.494471,0.116822
935,1982-11-30,10006,A C F INDUSTRIES INC,00080010,10,1,3743,Manuf,ACF,1010.0,...,-0.946207,-0.056846,1.161431,0.392651,-0.273396,-1.324789,1982,-0.157873,-0.522753,0.089235
936,1982-12-31,10006,A C F INDUSTRIES INC,00080010,10,1,3743,Manuf,ACF,1010.0,...,-0.434936,-0.110693,1.135611,0.385028,-0.483853,-1.321006,1982,-0.046802,-0.496126,0.026749
937,1983-01-31,10006,A C F INDUSTRIES INC,00080010,10,1,3743,Manuf,ACF,1010.0,...,-0.536237,-0.075222,1.185947,0.393112,-0.502099,-1.321311,1983,0.062964,-0.474563,0.096098
938,1983-02-28,10006,A C F INDUSTRIES INC,00080010,10,1,3743,Manuf,ACF,1010.0,...,-0.430000,-0.137022,1.130314,0.381271,-0.457126,-1.360672,1983,0.187878,-0.498308,0.058405
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2697838,2010-11-30,93433,MOTRICITY INC,62010710,11,3,9999,Other,MOTR,184994.0,...,4.171841,-0.234127,NaN,NaN,0.390336,0.372148,2010,NaN,-0.263696,0.026774
2697839,2010-12-31,93433,MOTRICITY INC,62010710,11,3,9999,Other,MOTR,184994.0,...,3.997548,-0.216713,NaN,NaN,0.412794,0.372490,2010,NaN,-0.217334,0.102048
2697840,2011-01-31,93433,MOTRICITY INC,62010710,11,3,9999,Other,MOTR,184994.0,...,3.763072,-0.213585,NaN,NaN,0.358546,0.378836,2011,1.109029,0.079833,0.000802
2697841,2011-02-28,93433,MOTRICITY INC,62010710,11,3,9999,Other,MOTR,184994.0,...,3.771236,-0.213850,NaN,NaN,0.358942,0.380040,2011,1.405542,-0.044231,0.030326


In [30]:
small_caps.to_csv('small_caps_cleaned.csv', index=False)

In [32]:
small_caps[variables].describe()

,agr,bm,mom12m,mve,operprof,roeq,absacc,acc,aeavol,age,...,sin,sp,std_dolvol,std_turn,stdcf,sue,tang,tb,turn,zerotrade
count,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,...,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05,5.520160e+05
mean,2.368407e-18,2.368407e-18,-9.267680e-19,-3.707072e-18,2.677330e-18,-5.045737e-18,-6.384402e-18,2.213946e-18,2.162459e-18,1.482829e-17,...,1.807198e-17,8.443886e-18,-2.883278e-18,-2.883278e-18,2.677330e-18,2.986252e-18,6.899273e-18,-7.208196e-19,1.729967e-17,1.647588e-18
std,1.000001e+00,1.000001e+00,9.901620e-01,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,...,4.949003e-01,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00
min,-2.477072e+00,-3.675209e+00,-2.407454e+00,-2.542340e+00,-5.956977e+00,-7.379423e+00,-1.731719e+00,-5.529644e+00,-1.704216e+00,-5.069287e+00,...,-4.808678e-01,-1.396991e+00,-2.564520e+00,-1.401770e+00,-1.726340e+00,-8.367131e+00,-4.060174e+00,-8.464058e+00,-1.724666e+00,-1.676580e+00
25%,-4.648360e-01,-6.926660e-01,-5.686881e-01,-7.129558e-01,-4.074640e-01,-2.470111e-01,-6.862418e-01,-3.031946e-01,-6.430368e-01,-8.125957e-01,...,0.000000e+00,-6.060303e-01,-7.273906e-01,-5.631064e-01,-4.420671e-01,-1.359609e-01,-4.852656e-01,-2.659098e-01,-6.311017e-01,-4.733552e-01
50%,-1.670352e-01,-1.362203e-02,-1.608896e-01,2.340682e-02,-6.247007e-02,6.660997e-02,-1.954041e-02,-4.870011e-02,-1.392564e-01,-4.474779e-02,...,0.000000e+00,-2.695718e-01,-6.271650e-02,-2.580713e-01,-2.317865e-01,3.917042e-02,1.418871e-02,-5.701691e-02,-2.338562e-01,-3.027817e-01
75%,5.948329e-02,4.465536e-01,3.525074e-01,8.070251e-01,1.679177e-01,3.301369e-01,2.626722e-01,4.485256e-01,1.775066e-01,7.559665e-01,...,0.000000e+00,2.563943e-01,5.326632e-01,8.072186e-02,1.650662e-01,2.103053e-01,4.262646e-01,3.357847e-01,2.162694e-01,-2.310803e-01
max,6.870841e+00,6.815459e+00,6.143388e+00,2.311101e+00,6.485250e+00,7.098563e+00,5.692694e+00,4.667655e+00,6.527347e+00,2.263181e+00,...,9.779768e+00,7.278875e+00,5.427177e+00,7.544868e+00,9.171982e+00,7.898903e+00,3.550130e+00,7.209866e+00,6.367616e+00,6.940018e+00
